In [1]:
import os, json

MODEL_PARAMS = {
    "flan-t5-small":   80e6,
    "flan-t5-base":    250e6,
    "flan-t5-large":   780e6,
    "flan-t5-xl":      3e9,
    "llama-2-7b-chat": 7e9,
}

# Three pairs spanning small/medium/large capability gap, mirroring the paper's own
# Figure 6 structure (small/medium/large quality-gap panels), drawn from the same 5
# models you already downloaded for the RL run -- no extra download, no GPU time needed.
PAIRS = [
    ("small_gap",  "flan-t5-xl",    "llama-2-7b-chat"),
    ("medium_gap", "flan-t5-large", "llama-2-7b-chat"),
    ("large_gap",  "flan-t5-small", "llama-2-7b-chat"),
]

N_TRAIN = 10000
N_TEST = 5000

WORK_DIR = os.path.expanduser("~/hybridllm_project")
OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_results = {}
for label, s_name, l_name in PAIRS:
    params_s, params_l = MODEL_PARAMS[s_name], MODEL_PARAMS[l_name]

    # Label-building: exact, deterministic -- every train query calls BOTH models regardless
    # of router quality (this is the paper's own method: r_det/r_prob/r_trans all need both
    # models' outputs to build a comparison label).
    label_building_cost = N_TRAIN * (params_s + params_l)

    # Test-deployment: depends on the (untrained) router's choices -- bounded.
    cost_best = N_TEST * params_s      # router always picks S
    cost_worst = N_TEST * params_l     # router always picks L
    cost_mid = N_TEST * (params_s + params_l) / 2   # neutral midpoint assumption

    total_best = label_building_cost + cost_best
    total_worst = label_building_cost + cost_worst
    total_mid = label_building_cost + cost_mid

    all_results[label] = {
        "small_model": s_name, "large_model": l_name,
        "params_s": params_s, "params_l": params_l,
        "label_building_cost_raw_params": label_building_cost,
        "total_cost_raw_params_best_case": total_best,
        "total_cost_raw_params_worst_case": total_worst,
        "total_cost_raw_params_midpoint": total_mid,
    }
    print(f"{label:>12} (S={s_name}, L={l_name}):")
    print(f"    label-building (exact):      {label_building_cost:>18,.0f}")
    print(f"    total, best/worst/mid:       {total_best:>18,.0f}  /  {total_worst:>18,.0f}  /  {total_mid:>18,.0f}")
    print()

# Save the full 3-pair comparison table
with open(os.path.join(OUTPUT_DIR, "hybridllm_pairs_comparison.json"), "w") as f:
    json.dump({"n_train": N_TRAIN, "n_test": N_TEST, "pairs": all_results}, f, indent=2)
print("saved " + os.path.join(OUTPUT_DIR, "hybridllm_pairs_comparison.json"))

# Also save the large_gap pair (biggest S-L span, most directly comparable to RL's full
# cheapest-to-priciest range) as hybrid_llm_results.json, in the format the RL notebook's
# comparison cell already expects, so that cell works unchanged.
primary = all_results["large_gap"]
compat_results = {
    "system": "hybridllm_paper_faithful_pair",
    "method": "analytical_no_generation",
    "note": ("Paper-faithful 2-model HybridLLM (S=flan-t5-small, L=llama-2-7b-chat), the "
             "largest-gap pair, chosen for direct comparability to RL's full cost range. "
             "See hybridllm_pairs_comparison.json for small/medium/large gap pairs."),
    "n_train": N_TRAIN, "n_test": N_TEST,
    "small_model": primary["small_model"], "large_model": primary["large_model"],
    "label_building_cost_raw_params": primary["label_building_cost_raw_params"],
    "total_cost_raw_params": primary["total_cost_raw_params_midpoint"],
    "total_cost_raw_params_best_case": primary["total_cost_raw_params_best_case"],
    "total_cost_raw_params_worst_case": primary["total_cost_raw_params_worst_case"],
    "total_cost_raw_params_extrapolated_15000": primary["total_cost_raw_params_midpoint"],  # already at 15k scale
    "full_scale_n": 15000, "scale_factor": 1.0,
}
with open(os.path.join(OUTPUT_DIR, "hybrid_llm_results.json"), "w") as f:
    json.dump(compat_results, f, indent=2)
print("saved " + os.path.join(OUTPUT_DIR, "hybrid_llm_results.json") + "  (large_gap pair, for RL's comparison cell)")


   small_gap (S=flan-t5-xl, L=llama-2-7b-chat):
    label-building (exact):      100,000,000,000,000
    total, best/worst/mid:       115,000,000,000,000  /  135,000,000,000,000  /  125,000,000,000,000

  medium_gap (S=flan-t5-large, L=llama-2-7b-chat):
    label-building (exact):      77,800,000,000,000
    total, best/worst/mid:       81,700,000,000,000  /  112,800,000,000,000  /  97,250,000,000,000

   large_gap (S=flan-t5-small, L=llama-2-7b-chat):
    label-building (exact):      70,800,000,000,000
    total, best/worst/mid:       71,200,000,000,000  /  105,800,000,000,000  /  88,500,000,000,000

saved /home/bhavana.ravi/hybridllm_project/outputs/hybridllm_pairs_comparison.json
saved /home/bhavana.ravi/hybridllm_project/outputs/hybrid_llm_results.json  (large_gap pair, for RL's comparison cell)
